# Gemini-Powered Stock News

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import yfinance as yf
from dotenv import load_dotenv

try:
    import google.generativeai as genai
except Exception:
    genai = None

load_dotenv(Path.cwd().parent / '.env')
API_KEY = os.getenv('GOOGLE_API_KEY', '').strip()
MODEL_NAME = os.getenv('GEMINI_MODEL', 'gemini-2.0-flash')

print('Gemini configured:', bool(API_KEY and genai))

Gemini configured: True


/Users/vmangla/Documents/Vaibhav/indian-stocks-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/y1/44jbvlv94_5c1bldzjp1gnlh0000gn/T/ipykernel_36873/3773601883.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
def normalize_ticker(stock_query: str) -> str:
    ticker = stock_query.strip().upper()
    if not ticker:
        return ticker
    if ticker.startswith('^') or ticker.endswith('.NS') or ticker.endswith('.BO'):
        return ticker
    return f'{ticker}.NS'

def fetch_stock_news(stock_query: str, limit: int = 30):
    ticker = normalize_ticker(stock_query)
    candidates = [ticker, stock_query.strip().upper(), f'{stock_query.strip().upper()}.BO']
    raw_news = []
    for candidate in candidates:
        if not candidate:
            continue
        try:
            items = yf.Ticker(candidate).news or []
            if items:
                raw_news = items
                ticker = candidate
                break
        except Exception:
            continue
    return ticker, raw_news[:limit]

def strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    return text.strip()

def extract_fields(items):
    cleaned = []
    for item in items:
        content = item.get('content', {}) if isinstance(item, dict) else {}
        provider = content.get('provider', {}) if isinstance(content, dict) else {}
        canonical_url = content.get('canonicalUrl', {}) if isinstance(content, dict) else {}
        click_url = content.get('clickThroughUrl', {}) if isinstance(content, dict) else {}
        cleaned.append({
            'title': content.get('title') or item.get('title', 'Untitled'),
            'source': provider.get('displayName') or content.get('provider') or item.get('publisher', 'Unknown'),
            'url': canonical_url.get('url') or click_url.get('url') or item.get('link', ''),
            'published_at': content.get('pubDate') or item.get('providerPublishTime', 'N/A'),
        })
    return cleaned

ticker, raw_news = fetch_stock_news('RELIANCE')
news_items = extract_fields(raw_news)
pd.DataFrame(news_items).head(10)

,title,source,url,published_at
0,How The Investment Story For Reliance Industri...,Simply Wall St.,https://finance.yahoo.com/markets/stocks/artic...,2026-04-03T14:07:15Z
1,India diesel exports to SE Asia hit 7-year hig...,Reuters,https://sg.finance.yahoo.com/news/india-diesel...,2026-04-01T01:51:53Z
2,OpenAI Appoints JioStar CEO to Lead Asia Expan...,GuruFocus.com,https://finance.yahoo.com/sectors/technology/a...,2026-03-26T12:28:31Z
3,Exclusive-India's Reliance buys 5 million barr...,Reuters,https://sg.finance.yahoo.com/news/exclusive-in...,2026-03-24T10:06:43Z
4,Factbox-Ambani's Reliance Jio: businesses and ...,Reuters,https://sg.finance.yahoo.com/news/factbox-amba...,2026-03-23T09:38:55Z
5,India Coca‑Cola bottler SLMG says Middle East ...,Reuters,https://sg.finance.yahoo.com/news/india-coca-c...,2026-03-23T04:53:55Z
6,How The Reliance Industries (NSEI:RELIANCE) St...,Simply Wall St.,https://finance.yahoo.com/markets/stocks/artic...,2026-03-20T10:06:50Z
7,"Factbox-India, a major energy consumer and ref...",Reuters,https://sg.finance.yahoo.com/news/factbox-indi...,2026-03-19T08:46:55Z
8,"India orders oil, gas firms to share import, e...",Reuters,https://sg.finance.yahoo.com/news/india-orders...,2026-03-18T19:52:15Z
9,"Reliance Industries, Samsung subsidiary sign $...",ESG Dive,https://www.esgdive.com/news/reliance-industri...,2026-03-18T10:37:00Z


In [3]:
if API_KEY and genai and news_items:
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)
    prompt = f'''
You are a market news analyst. Select the 10 most important headlines for the stock below and summarize each item briefly for investors.
Use only the provided items and do not invent titles, sources, URLs, or publication times.
Return valid JSON only with this exact structure:
{{"summary": string, "highlights": [{{"title": string, "source": string, "url": string, "published_at": string, "summary": string}}]}}

Stock: {ticker}

News items: {json.dumps(news_items, ensure_ascii=False, indent=2)}
'''
    response = model.generate_content(prompt)
    print(getattr(response, 'text', ''))
else:
    print('Add GOOGLE_API_KEY to .env, then rerun this notebook.')

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 40.453409626s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quo